# 03 — Оценка checkpoint: два отдельных протокола

A: окна вокруг известных отметок. B: полный проход видео. Обучение и подбор порога не выполняются.

## Как интерпретировать результат
Протокол A — классификация окон вокруг **известных отметок**, а не автоматический поиск события во всём видео.
Согласованный `within_room_unique_video` split: 30 train / 10 validation; точные копии не пересекаются.
Общий кабинет, люди и неизвестные сеансы могут пересекаться. Независимость камер и сеансов не подтверждена.
Validation используется для выбора checkpoint по loss, а не как независимый финальный тест.
Первый smoke-run предсказывал нарушение для всех validation-окон (accuracy 0.5, recall 1.0).
Это проверка работоспособности, не хорошее качество модели и не законченный детектор.
Исторические метрики и provenance сохранены в `experiments/r3d18_smoke_result.md` и прежних `runs/`.
Протокол B использует только видео для предсказаний; разметка применяется после прохода для оценки.


In [ ]:
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "src/overfitted_underqualified").is_dir() and (p / "notebooks").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Запустите из корня репозитория или notebooks/")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import os
import time
from datetime import datetime, timezone
from uuid import uuid4

import pandas as pd
from IPython.display import FileLink, display
from dotenv import dotenv_values

from overfitted_underqualified.ml.evaluate import (
    archive_evaluation_report,
    evaluate_prepared,
    prepare_evaluation,
    resolve_checkpoint,
    save_evaluation_report,
)
from overfitted_underqualified.ml.plots import plot_confusion, show_windows

import yaml
import matplotlib.pyplot as plt

evaluation_settings = {**dotenv_values(ROOT / ".env"), **os.environ}
EVAL_PROTOCOL = evaluation_settings.get("EVAL_PROTOCOL") or "annotation_windows"
if EVAL_PROTOCOL not in {"annotation_windows", "full_video", "both", "actions", "primitives", "banknotes"}:
    raise ValueError("EVAL_PROTOCOL: annotation_windows, full_video, both, actions, primitives or banknotes")
RUN_A = EVAL_PROTOCOL in {"annotation_windows", "both"}
RUN_B = EVAL_PROTOCOL in {"full_video", "both"}
print("Протокол:", EVAL_PROTOCOL)


## 1. Checkpoint и сохранённая конфигурация

Показываем доступные `runs/*/best.pt`. Если файл ровно один, выбираем его автоматически.
Если их несколько, укажите `CHECKPOINT_PATH=runs/<нужный запуск>/best.pt` в корневом `.env`
или переменной окружения. Явный неверный путь вызывает ошибку, без незаметной подмены модели.
На локальном компьютере доступны только локальные файлы: скачайте папку запуска с Brev
или выполняйте оценку на Brev. Содержимое данных сверяется с хешами checkpoint.


In [ ]:
# Local .env values can be overridden by environment variables in this kernel.
evaluation_settings = {**dotenv_values(ROOT / ".env"), **os.environ}
available_checkpoints = sorted((ROOT / "runs").glob("*/best.pt"))
display(pd.DataFrame({"available_checkpoint": [str(p.relative_to(ROOT)) for p in available_checkpoints]}))
checkpoint_path = resolve_checkpoint(evaluation_settings.get("CHECKPOINT_PATH"), root=ROOT)
print("Выбран checkpoint:", checkpoint_path)
if RUN_A:
    preparation_start = time.perf_counter()
    evaluation = prepare_evaluation(
        checkpoint_path, device=(evaluation_settings.get("EVAL_DEVICE") or "cpu"),
        events_path=ROOT / "data/annotations/events.csv",
        split_path=ROOT / "data/processed/split_proposal.csv",
        video_dir=Path((evaluation_settings.get("VIDEO_DIR") or str(ROOT / "data/raw/videos"))),
    )
    if evaluation["device"].type == "cuda":
        import torch
        torch.cuda.synchronize(evaluation["device"])
    preparation_sec = time.perf_counter() - preparation_start
    display(evaluation["checkpoint"]["config"])
    print("Checkpoint:", checkpoint_path, "epoch:", evaluation["checkpoint"]["epoch"])
    print("Проверено validation-окон:", len(evaluation["dataset"]))


## 2. Оценка validation

Модель работает в eval mode: параметры и BatchNorm не обновляются. Precision/recall/F1 относятся к violation=1. Accuracy считается по всем окнам. Оценка не перезаписывает результаты обучения.

In [ ]:
if RUN_A:
    import time

    import torch

    from overfitted_underqualified.ml.evaluate import aggregate_video_predictions

    if evaluation["device"].type == "cuda":
        torch.cuda.synchronize(evaluation["device"])
    processing_start = time.perf_counter()
    metrics, rows = evaluate_prepared(evaluation)
    if evaluation["device"].type == "cuda":
        torch.cuda.synchronize(evaluation["device"])
    processing_sec = time.perf_counter() - processing_start
    predictions = pd.DataFrame(rows)
    display(pd.Series({key: value for key, value in metrics.items() if key != "confusion_matrix"}, name="validation"))
    plot_confusion(metrics)
    display(predictions)

    # Distinguish window metrics above from canonical-video metrics below.
    video_metrics, video_rows = aggregate_video_predictions(rows)
    video_predictions = pd.DataFrame(video_rows)
    display(pd.Series({key: value for key, value in video_metrics.items() if key != "confusion_matrix"}, name="video-level validation"))
    plot_confusion(video_metrics)
    display(video_predictions)
    print(f"Оценка окон: {processing_sec:.3f} с; среднее {processing_sec / len(video_rows):.3f} с/видео.")
    print("Включены decode, preprocessing, transfer, inference и оконные метрики; загрузка checkpoint исключена.")
    print("Это время выбранных окон, не полного просмотра видео; warm-up не выполнялся.")


## 3. Просмотр ошибок

Показываем до шести ошибочно классифицированных окон с фиксированным порогом. Это разбор ошибок, а не изменение разметки или настройка порога.

In [ ]:
if RUN_A:
    errors = predictions.loc[predictions["label"].ne(predictions["prediction"])]
    display(errors)
    selected = errors.head(6)
    show_windows(evaluation["dataset"], selected.index,
                 [f"{row.clip_id}; y={row.label}, pred={row.prediction}, p={row.probability:.2f}"
                  for row in selected.itertuples()])
    print("Всего ошибок:", len(errors), "из", len(predictions))


Следующий эксперимент должен проверять конкретную гипотезу. Для вывода о поиске событий нужен отдельный full-video протокол; для переноса на другие камеры/сеансы — независимые данные.

## 4. False positives и false negatives
Разделяем лишние тревоги и пропуски. Для каждого окна доступны исходные ID и время центра.
Показываем обе группы отдельно: пустая группа означает отсутствие таких ошибок на этой выборке,
а не доказательство безошибочности. Кадры — визуальные свидетельства для исследования, не юридическое подтверждение.

In [ ]:
if RUN_A:
    error_analysis = predictions.copy()
    error_analysis["outcome"] = [
        "TP" if truth == pred == 1 else "TN" if truth == pred == 0 else "FP" if pred == 1 else "FN"
        for truth, pred in zip(error_analysis["label"], error_analysis["prediction"])
    ]
    error_analysis["source_ids"] = [list(record.aliases) for record, _ in evaluation["dataset"].samples]
    display(error_analysis.groupby("outcome").agg(windows=("clip_id", "size"), mean_probability=("probability", "mean")))
    for outcome in ("FP", "FN"):
        subset = error_analysis.loc[error_analysis["outcome"].eq(outcome)]
        print(outcome, "окон:", len(subset))
        display(subset)
        selected = subset.head(3)
        show_windows(evaluation["dataset"], selected.index,
                     [f"{outcome}: {row.clip_id}; p={row.probability:.2f}" for row in selected.itertuples()])


## A.5. Временная диагностика: что измеряется сейчас
В протоколе A модель выдаёт вероятность класса окна, **не predicted timestamp**. Поэтому timestamp MAE,
ошибка локализации и доля событий в допуске в A **N/A**, а не ноль.
`center_sec` задан разметкой для положительных окон; сравнивать его с разметкой как предсказание нельзя.
Ниже проверяем только выборку: границы окна, ближайший реально выбранный кадр и его отклонение
от отметки. Это погрешность дискретизации кадров, не ошибка модели. Для FP истинной отметки нет.
Для настоящей оценки локализации нужны full-video inference и заранее заданное сопоставление событий,
включая обработку лишних и пропущенных событий. Контракт FastAPI `Finding.timecode_sec` сохраняется;
он не заполняется центрами известных окон в качестве якобы найденных событий.

In [ ]:
if RUN_A:
    import numpy as np

    from overfitted_underqualified.ml.dataset import sample_times

    timing_rows = []
    dataset = evaluation["dataset"]
    for index, (record, center) in enumerate(dataset.samples):
        requested = sample_times(center, record.last_time, dataset.window_sec, dataset.frames)
        sampled = np.clip(np.rint(requested * record.fps), 0, record.frame_count - 1) / record.fps
        reference = center if record.label == 1 else np.nan
        nearest = sampled[np.argmin(abs(sampled - reference))] if record.label == 1 else np.nan
        timing_rows.append({"clip_id": record.clip_id, "outcome": error_analysis.loc[index, "outcome"],
            "reference_sec": reference, "window_start_sec": sampled.min(), "window_end_sec": sampled.max(),
            "nearest_sample_sec": nearest, "sampling_offset_sec": abs(nearest-reference),
            "predicted_timestamp_sec": np.nan, "timestamp_error_sec": np.nan})
    timing = pd.DataFrame(timing_rows)
    display(timing)
    display(timing["sampling_offset_sec"].describe())
    print("Timestamp localization metrics: N/A — no model timestamp predictions")


## 6. Выводы и следующий исследовательский шаг
Сначала прочитайте итог ниже для выбранного checkpoint. Затем запишите наблюдения по FP/FN:
видно ли действие после crop, хватает ли временного контекста, похожи ли обычные действия на нарушения.
Это гипотезы для следующего эксперимента, а не автоматическая коррекция источника меток.

In [ ]:
if RUN_A:
    counts = error_analysis["outcome"].value_counts()
    print(f"Validation: {len(predictions)} окон; FP={counts.get('FP', 0)}, FN={counts.get('FN', 0)}.")
    print(f"Accuracy={metrics['accuracy']:.3f}, precision={metrics['precision']:.3f}, recall={metrics['recall']:.3f}, F1={metrics['f1']:.3f}.")
    if predictions["prediction"].nunique() == 1:
        print("Модель выдаёт один класс для всех окон. Проверьте дисбаланс предсказаний; recall отдельно недостаточен.")
    print("Локализация не оценена. Независимость камер/сеансов не подтверждена; это не готовый detector.")


Перед коммитом очистите outputs, execution counts и attachments. Не переносите изображения,
описания событий или индивидуальные предсказания в публичные отчёты. Сохранённые runs остаются локальными.

## 7. Автоматическое сохранение и архив для скачивания
Следующая ячейка сохраняет уже рассчитанные результаты без повторного inference.
Каждый запуск создаёт новый каталог `reports/evaluation_<run>_<UTC>_<id>/` и ZIP рядом с ним.
В архив входят метрики, предсказания, FP/FN, временная диагностика, confusion matrix,
provenance и доступные артефакты обучения при совпадении манифеста.
Веса, исходные видео, банковские кадры и `.env` в архив не включаются.
Скачайте ZIP по ссылке в выводе ячейки либо через файловый браузер Jupyter в `reports/`.
После этого архив можно прикрепить в чат для разбора результатов.


In [ ]:
if RUN_A:
    report_name = (
        f"evaluation_{checkpoint_path.parent.name}_"
        f"{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}_{uuid4().hex[:8]}"
    )
    report_dir = ROOT / "reports" / report_name
    saved_report = save_evaluation_report(
        evaluation, checkpoint_path, report_dir, metrics, rows,
        processing_sec=processing_sec, preparation_sec=preparation_sec,
    )
    error_analysis.to_csv(report_dir / "error_analysis.csv", index=False)
    timing.to_csv(report_dir / "sampling_diagnostics.csv", index=False)
    archive_path = archive_evaluation_report(report_dir, checkpoint_path)
    print("Результаты:", report_dir)
    print("Архив для отправки:", archive_path)
    display(FileLink(os.path.relpath(archive_path, Path.cwd()), result_html_prefix="Скачать ZIP: "))


## B.1. Полное видео: фиксированный протокол
Выбор выше: EVAL_PROTOCOL=full_video или both. Для B настройки читаются до предсказаний из configs/full_video.yaml.
Порог 0.5, шаг 2 с, допуск сопоставления 5 с — исследовательские значения, не подобранные на validation.
Оцениваем все 10 канонических validation-видео. Разные отметки одного видео остаются отдельными событиями.
Не загружаем pretrained-веса и не обновляем BatchNorm. Подготовка сверяет checkpoint, SHA-256, split и метки.


In [ ]:
from overfitted_underqualified.ml.full_video_evaluation import (
    prepare_full_evaluation, evaluate_full_prepared, save_full_report, archive_full_report,
)
from overfitted_underqualified.ml.inference import ScanConfig, evidence_frames
from overfitted_underqualified.ml.plots import show_evidence

if RUN_B:
    full_config = yaml.safe_load((ROOT / (evaluation_settings.get("FULL_VIDEO_CONFIG") or "configs/full_video.yaml")).read_text(encoding="utf-8"))
    display(full_config)
    full_prepared = prepare_full_evaluation(
        checkpoint_path, ScanConfig(**full_config["scan"]),
        device=evaluation_settings.get("EVAL_DEVICE") or "cpu",
        events_path=ROOT / "data/annotations/events.csv", split_path=ROOT / "data/processed/split_proposal.csv",
        video_dir=Path(evaluation_settings.get("VIDEO_DIR") or str(ROOT / "data/raw/videos")),
    )
    display(pd.DataFrame([{"clip_id": r.clip_id, "label": r.label, "sha256": r.sha256}
                          for r in full_prepared["records"]]))


In [ ]:
if RUN_B:
    full_result = evaluate_full_prepared(full_prepared, tolerance_sec=full_config["matching_tolerance_sec"])
    full_summary = full_result["summary"]
    full_videos = pd.DataFrame(full_summary["videos"])
    display(full_videos)
    display(pd.Series(full_summary["metrics"]["video"], name="B: full-video"))
    plot_confusion(full_summary["metrics"]["video"], threshold=full_config["scan"]["threshold"])
    display(pd.Series(full_summary["metrics"]["false_alarms"], name="Тревоги на отрицательных видео"))
    print("Полный проход, секунд:", full_result["processing_sec"])
    if RUN_A:
        display(pd.DataFrame([
            {"protocol": "A: annotation-centred canonical video aggregation", **video_metrics},
            {"protocol": "B: full-video scan", **full_summary["metrics"]["video"]},
        ]))
        print("Разный охват окон. Разность F1 не является измерением улучшения одной и той же задачи.")


## B.2. Временное сопоставление и пропуски
Алгоритм максимизирует число пар в заданном допуске, затем минимизирует суммарную абсолютную ошибку.
Каждая отметка и кандидат используются максимум один раз. Лишний кандидат возле уже сопоставленной отметки
остаётся несопоставленным. Для положительных видео это unresolved, а не подтверждённый FP: разметка неполна.
MAE относится только к matched-парам — читайте его вместе с числом пропущенных отметок.


In [ ]:
if RUN_B:
    temporal_matches = pd.DataFrame(full_summary["matching"])
    display(pd.Series(full_summary["metrics"]["temporal"], name="Предварительно относительно известных отметок"))
    display(temporal_matches)
    display(temporal_matches.loc[temporal_matches["status"].ne("matched")])
    matched = temporal_matches.loc[temporal_matches["status"].eq("matched")]
    if not matched.empty:
        matched.plot.bar(x="clip_id", y="signed_error_sec", ylabel="Ошибка предполагаемого времени, с")
        plt.show()
        plt.close("all")


## B.3. Ошибки и локальные кадры
Сначала рассмотрите video-level FP/FN. В выбранном видео видны вероятность, поддержки кандидатов и известные
отметки (они добавлены только на график после inference). Для пропуска без кандидатов показывается контекст
исходной отметки, явно как reference. Это не предсказанное время. Кадры не сохраняются в отчёт/ZIP.


In [ ]:
if RUN_B:
    full_errors = full_videos.loc[full_videos["outcome"].isin(["FP", "FN"])]
    display(full_errors)
    SELECTED_VIDEO = full_errors.iloc[0]["clip_id"] if not full_errors.empty else full_videos.iloc[0]["clip_id"]
    record = next(r for r in full_prepared["records"] if r.clip_id == SELECTED_VIDEO)
    scan = next(s for s in full_result["scans"] if s["clip_id"] == SELECTED_VIDEO)
    trace = pd.DataFrame(scan["windows"])
    ax = trace.plot(x="center_sec", y="probability", figsize=(12, 3), ylim=(0, 1), marker=".")
    ax.axhline(full_config["scan"]["threshold"], color="red", linestyle="--")
    for c in scan["candidates"]:
        ax.axvspan(c["start_sec"], c["end_sec"], color="orange", alpha=0.2)
        ax.axvline(c["timestamp_sec"], color="orange")
    for reference in record.event_times:
        ax.axvline(reference, color="green", linestyle=":")
    ax.set_title(f"{SELECTED_VIDEO}: orange=predicted candidate; green=known reference")
    plt.show()
    plt.close("all")
    display(pd.DataFrame(scan["candidates"]))
    for candidate in scan["candidates"][:3]:
        show_evidence(evidence_frames(record.path, candidate["timestamp_sec"]),
                      f"{SELECTED_VIDEO}: контекст кандидата {candidate['candidate_id']}, тип неизвестен")
    if not scan["candidates"] and record.event_times:
        show_evidence(evidence_frames(record.path, min(record.event_times[0], record.last_time)),
                      f"{SELECTED_VIDEO}: REFERENCE пропущенной отметки, не предсказание")


## B.4. Вывод и архив
Есть бинарный поиск кандидатов с предполагаемыми временами. Квалификация действия и юридического нарушения
не выполнена. Общая комната, люди и неизвестные сеансы ограничивают переносимость оценки; checkpoint ранее
выбирался по этой validation. Наблюдения по кадрам — гипотезы, не изменения меток.
Следующая ячейка автоматически сохраняет уже полученные результаты и ZIP по allowlist: без банковских кадров,
видео, весов и .env. Для отдельного сравнения запусков сохраняйте разные протоколы и конфигурации.


In [ ]:
if RUN_B:
    full_report_dir = ROOT / "reports" / f"evaluation_full_{checkpoint_path.parent.name}_{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}_{uuid4().hex[:8]}"
    full_report = save_full_report(full_prepared, full_result, full_report_dir)
    full_archive = archive_full_report(full_report_dir)
    temporal = full_report["metrics"]["temporal"]
    print(f"Совпало {temporal['matched']} из {temporal['annotations']} известных отметок; пропущено {temporal['unmatched_annotations']}.")
    print("Несопоставленных кандидатов:", temporal["unmatched_predictions"], "; MAE только matched:", temporal["matched_mae_sec"])
    print("Отчёт:", full_report_dir)
    display(FileLink(os.path.relpath(full_archive, Path.cwd()), result_html_prefix="Скачать полный проход ZIP: "))


## C. Отдельная оценка распознавателя действий и полной цепочки
Укажите `EVAL_PROTOCOL=actions`, `RUN_ACTION_EVAL=true`, `ACTION_CHECKPOINT_PATH` и
`ACTION_ENABLED_TYPES` (через запятую). Типы должны иметь train-примеры и положительные/отрицательные
validation-примеры. Сейчас checkpoint отсутствует: полнота action-разметки не проверена.
Оценка не обучает модель, не меняет BatchNorm и не подбирает порог.
Показываем отдельные 2×2 confusion matrix каждого доступного класса; multiclass-матрица здесь неверна.
Допуск времени — 5 секунд, задан заранее. Бинарный gate может пропустить действие до распознавателя.


In [ ]:
from overfitted_underqualified.ml.action_train import read_config
from overfitted_underqualified.ml.action_evaluation import evaluate_actions

RUN_ACTION_EVAL = str(evaluation_settings.get("RUN_ACTION_EVAL", "false")).lower() == "true"
action_report = None
if RUN_ACTION_EVAL:
    action_path_value = evaluation_settings.get("ACTION_CHECKPOINT_PATH")
    if not action_path_value:
        raise ValueError("Укажите ACTION_CHECKPOINT_PATH реального обученного распознавателя")
    enabled_types = [x.strip() for x in evaluation_settings.get("ACTION_ENABLED_TYPES", "").split(",") if x.strip()]
    action_eval_config = read_config(ROOT / evaluation_settings.get("ACTION_EVAL_CONFIG", evaluation_settings.get("ACTION_CONFIG", "configs/actions.yaml")))
    action_eval_config["device"] = evaluation_settings.get("ACTION_DEVICE", "cpu")
    binary_path = resolve_checkpoint(evaluation_settings.get("CHECKPOINT_PATH"), root=ROOT)
    action_report_dir, action_report = evaluate_actions(
        action_eval_config, binary_path, ROOT / action_path_value, enabled_types, tolerance_sec=5.0)
    display(pd.DataFrame(action_report["window_metrics"]["per_class"]))
    display(pd.Series(action_report["window_metrics"]["macro"], name="Macro по оцениваемым классам"))
    for item in action_report["window_metrics"]["per_class"]:
        if item["evaluable"]:
            fig, ax = plt.subplots(figsize=(4, 3))
            ax.imshow(item["confusion_matrix"], cmap="Blues")
            ax.set(title=item["event_type"], xlabel="Предсказание", ylabel="Проверенная метка",
                   xticks=[0, 1], yticks=[0, 1], xticklabels=["Нет", "Есть"], yticklabels=["Нет", "Есть"])
            for y, row in enumerate(item["confusion_matrix"]):
                for x, value in enumerate(row): ax.text(x, y, str(value), ha="center", color="red")
            plt.show()
            plt.close(fig)
    display(pd.DataFrame(action_report["temporal"]["per_class"]))
    display(pd.DataFrame(action_report["temporal"]["errors"]))
    display(pd.Series({k: action_report[k] for k in (
        "device", "full_video_processing_sec", "candidate_windows", "abstentions", "recognition_coverage")}))
    display(FileLink(str(action_report_dir.with_suffix(".zip"))))
else:
    print("Оценка типов выключена. Проверки со stub не являются качеством модели.")


### Интерпретация и дальнейшая проверка
`null` означает отсутствие основания для метрики. Неоценимые классы не входят в macro;
число вошедших классов сохранено в отчёте. Неопределённые кандидаты требуют просмотра.
Точечные метки не гарантируют полноту всех действий на всём положительном видео:
непарные предсказания остаются неразрешёнными, а full-video precision/F1 не выдумываются.
Архив содержит численные результаты и provenance, без кадров. Для просмотра доказательств
используйте локальный `evidence_frames` из предыдущего раздела. Общий кабинет и неизвестные
сеансы не позволяют объявить оценку независимой. Исторический бинарный baseline сохранён.


## X-18 / X-2: один тип и проверка геометрии
В X-2 предмет на столе — банкнота по уточнению владельца. Метки и split не менялись.
Портретное видео 464×848 искажалось resize 171×128 и crop 112×112.
Сравнили один тип с прежней геометрией и ROI кассира/клиента, выбранным по train.
Конфигурации: `configs/actions_identity_control.yaml` и `configs/actions_identity_roi.yaml`.
Обе: 30 фиксированных эпох, train-only балансировка, порог 0.7. Frozen-признаки
вычисляются один раз на запуск; validation не участвует в градиентах.
Полный отчёт: `reports/identity_geometry_20261018.md`. Ниже только чтение результатов.


In [ ]:
import json
identity_path = ROOT / "reports/identity_geometry_20261018.json"
identity_study = json.loads(identity_path.read_text(encoding="utf-8")) if identity_path.exists() else None
if identity_study:
    comparison = pd.DataFrame([{
        "variant": r["name"], "train_F1": r["train"]["f1"],
        "validation_F1": r["validation"]["f1"], "X2_score": r["x2_score"],
        "FP_windows": r["validation"]["confusion_matrix"][0][1],
        "FN_windows": r["validation"]["confusion_matrix"][1][0],
        "matched_events": r["temporal"]["matched"], "missed_events": r["temporal"]["missed"]
    } for r in identity_study["results"]]).set_index("variant")
    display(comparison)
else:
    print("Отчёт сравнения пока отсутствует; обучение автоматически не запускается.")


In [ ]:
if identity_study:
    display(pd.DataFrame([{"variant": r["name"], **r["temporal"]} for r in identity_study["results"]]))
    for result in identity_study["results"]:
        print(result["name"], "checkpoint:", result["checkpoint_sha256"])
        archive = ROOT / (result["evaluation"] + ".zip")
        if archive.exists():
            display(FileLink(str(archive)))


Оба варианта выучили train (F1=1), но пропустили X-2: validation F1=0.
ROI исправляет геометрию, однако прирост распознавания не подтверждён.
Нужны дополнительные независимые положительные видео и проверяемые отрицательные
примеры идентификации. Из одной точки нельзя получить независимые данные
нарезкой соседних кадров. Validation уже использовалась для исследования;
кабинет, люди и неизвестные сеансы могут пересекаться. Основной сервис не переключён.


## Проверка конкретного типа: unsecured_accepted_cash
После визуального разбора X-38 добавлена одна положительная action-метка в validation.
Для X-2/X-31/X-39 проверено отсутствие этого типа в окне. Неясные X-10/X-14/X-16/X-34
маскированы для этого типа, а не считаются отрицательными. Бинарные метки, время и split сохранены.
Конфигурация: `configs/actions_unsecured_cash.yaml`, фиксированные 30 эпох и порог 0.7.
Новые метки получены по кадровым последовательностям в этой сессии; независимой экспертизы нет.


In [ ]:
import json
cash_path = ROOT / "reports/unsecured_cash_20261018.json"
if cash_path.exists():
    cash_result = json.loads(cash_path.read_text(encoding="utf-8"))
    display(pd.DataFrame(cash_result["known_windows"]).set_index("clip_id"))
    display(pd.Series(cash_result["window_metrics"], name="Known validation windows"))
    display(pd.DataFrame([cash_result["temporal"]]))
    display(pd.DataFrame(cash_result["posthoc_error_review"]))
    print("Положительный и отрицательный пример правильно распознаны:",
          cash_result["useful_window_recognition_demonstrated"])
    print("Полный отчёт:", cash_result["evaluation"], "checkpoint:", cash_result["checkpoint_sha256"])
else:
    print("Отчёт ещё не создан; обучение автоматически не запускается.")


Измерение на одном положительном и трёх отрицательных окнах: [[TN,FP],[FN,TP]]=[[3,0],[1,0]], F1=0.
X-38 пропущен; X-2 даёт ложную тревогу около 4 с при полном проходе.
Score X-2 выше X-38: одним снижением порога правильного разделения не получить.
Это проверка с отрицательным результатом, не положительная приёмка P0.
Полная временная разметка отсутствует: window-метрики не заменяют full-video precision/F1.
Нужны дополнительные независимые положительные видео и проверка условий хранения.


## Видимые признаки и временная цепочка
Для запуска только этого раздела задайте `EVAL_PROTOCOL=primitives`. По умолчанию читаем сохранённые
результаты. `RUN_PRIMITIVE_EVAL=true` явно включает новую оценку checkpoint на всех десяти validation-видео.
Сканер получает только видео, проходит его с начала и не использует бинарный gate.

In [ ]:
import os, json
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from dotenv import dotenv_values
from overfitted_underqualified.ml.primitive_model import read_config as read_primitive_config
primitive_settings = {**dotenv_values(ROOT / '.env'), **os.environ}
primitive_config = read_primitive_config(ROOT / primitive_settings.get('PRIMITIVE_CONFIG', 'configs/primitives_multiview.yaml'))
primitive_run = Path(primitive_config['output_dir'])
display(pd.Series({key: primitive_config[key] for key in
    ('seed', 'window_sec', 'frames', 'epochs', 'threshold', 'sampling', 'roi', 'scan_step_sec')}))

from overfitted_underqualified.ml.primitive_evaluation import evaluate_primitives
from IPython.display import FileLink
primitive_checkpoint = ROOT / primitive_settings.get('PRIMITIVE_CHECKPOINT_PATH', str(primitive_run / 'last.pt'))
primitive_report = ROOT / primitive_settings.get('PRIMITIVE_REPORT_DIR', 'reports/evaluation_primitives_multiview_20261018T013815Z')
if primitive_settings.get('RUN_PRIMITIVE_EVAL', 'false').lower() == 'true':
    primitive_report = ROOT / 'reports' / ('evaluation_primitives_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
    primitive_result = evaluate_primitives(primitive_config, primitive_checkpoint, primitive_report,
        device=primitive_settings.get('PRIMITIVE_DEVICE', primitive_config['device']))
elif (primitive_report / 'summary.json').exists():
    primitive_result = json.loads((primitive_report / 'summary.json').read_text(encoding='utf-8'))
else:
    primitive_result = None
    print('Сохранённого отчёта нет:', primitive_report)

In [ ]:
if primitive_result:
    primitive_metrics = pd.DataFrame(primitive_result['primitive_metrics']['per_class'])
    display(primitive_metrics[['event_type','support','known_negatives','unknown','precision','recall','f1']])
    fig, axes = plt.subplots(1,len(primitive_metrics), figsize=(12,3))
    for ax, (_,row) in zip(axes,primitive_metrics.iterrows()):
        matrix = row['confusion_matrix']; ax.imshow(matrix,cmap='Blues')
        for y in range(2):
            for x in range(2): ax.text(x,y,str(matrix[y][x]),ha='center',va='center')
        ax.set(xticks=[0,1],yticks=[0,1],xlabel='Prediction',ylabel='Target',title=row['event_type'])
    plt.tight_layout(); plt.show()
    display(pd.DataFrame(json.loads((primitive_report/'primitive_errors.json').read_text(encoding='utf-8'))))
    display(pd.Series(primitive_result['temporal']))
    display(pd.DataFrame(primitive_result['videos']))

In [ ]:
# Change clip_id to inspect another validation video; no model execution here.
primitive_clip_id = 'X-38'
primitive_scan_path = primitive_report / f'{primitive_clip_id}_scan.json'
if primitive_scan_path.exists():
    primitive_scan = json.loads(primitive_scan_path.read_text(encoding='utf-8'))
    trajectory = pd.json_normalize(primitive_scan['observations'])
    fig, ax = plt.subplots(figsize=(12,4))
    for name in ('client_at_counter','banknotes_on_counter','cash_handover_to_cashier'):
        ax.plot(trajectory['time_sec'],trajectory[f'probabilities.{name}'],label=name)
    report_config = json.loads((primitive_report/'config.json').read_text(encoding='utf-8'))
    ax.axhline(report_config['sequence']['positive_threshold'],color='gray',linestyle='--')
    ax.axhline(report_config['sequence']['negative_threshold'],color='gray',linestyle=':')
    for event in primitive_scan['events']: ax.axvline(event['timestamp_sec'],color='red',alpha=.4)
    ax.set(xlabel='Seconds (causal endpoint)',ylabel='Score',ylim=(0,1),title=primitive_clip_id); ax.legend(); plt.show()
    display(pd.DataFrame(primitive_scan['events']))
    display(pd.DataFrame(primitive_scan['unresolved']))
if primitive_report.with_suffix('.zip').exists():
    display(FileLink(str(primitive_report.with_suffix('.zip'))))

In [ ]:
# Optional private evidence frames for known FP/FN; no inference or label edits.
if primitive_result and primitive_settings.get('SHOW_PRIMITIVE_ERROR_FRAMES','false').lower()=='true':
    from overfitted_underqualified.ml.primitive_data import prepare_primitive_data, causal_times
    from overfitted_underqualified.ml.dataset import decode_frames
    from overfitted_underqualified.ml.primitive_model import read_config as read_primitive_config
    evidence_config = read_primitive_config(primitive_report / 'config.json')
    evidence_records, _, _ = prepare_primitive_data(evidence_config)
    record_by_id = {r.clip_id:r for r in evidence_records}
    known_errors = json.loads((primitive_report/'primitive_errors.json').read_text(encoding='utf-8'))[:3]
    if known_errors:
        fig, axes = plt.subplots(len(known_errors),3,figsize=(12,5*len(known_errors)),squeeze=False)
        for i,error in enumerate(known_errors):
            record = record_by_id[error['clip_id']]
            times = causal_times(error['time_sec'],record.last_time,evidence_config['window_sec'])[[0,7,15]]
            frames = decode_frames(record,times)
            for ax,frame,stamp in zip(axes[i],frames,times):
                ax.imshow(frame.permute(1,2,0).numpy()); ax.axis('off')
                ax.set_title(f"{error['clip_id']} {stamp:.2f}s\n{error['primitive']} {error['error']}",fontsize=9)
        plt.tight_layout(); plt.show()
else:
    print('Кадры ошибок скрыты; SHOW_PRIMITIVE_ERROR_FRAMES=true включает локальный просмотр.')


### Ограничения вывода
Статус `research_candidate` означает гипотезу для проверки. Эта цепочка не отслеживает тождество
купюр и не устанавливает завершение операции или юридическую применимость. `insufficient_data`
не означает «нарушений нет». Precision/F1 всего видео остаются неопределёнными без полной разметки.
Сравнивайте метрики примитивов отдельно от старой action-модели и бинарного baseline.
Архив содержит численные результаты и provenance, без кадров и весов. Перед Git очистите outputs.

## Детектор банкнот + временная история
  `EVAL_PROTOCOL=banknotes` пропускает бинарные протоколы A/B. Ниже по умолчанию читается
  сохранённый отчёт; `RUN_BANKNOTE_EVAL=true` выполняет новый проход 10 validation-видео.
  `BANKNOTE_TRACKING=true` добавляет проверку изображения уже найденного объекта между
  детекциями. Score сопровождения не является калиброванной вероятностью модели.
  Отдельно показаны примитивы и визуальная последовательность на четырёх просмотренных
  сетках кадров. Эти четыре видео и прежняя validation уже использовались для диагностики;
  это не независимый test и не проверка всех юридических нарушений.


In [ ]:
import os,json
from pathlib import Path
from datetime import datetime,timezone
from dotenv import dotenv_values
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display,FileLink
from overfitted_underqualified.ml.banknote_evaluation import evaluate_banknote_pipeline
banknote_settings = {**dotenv_values(ROOT / '.env'), **os.environ}
banknote_report = ROOT / banknote_settings.get('BANKNOTE_REPORT_DIR','reports/evaluation_banknotes_20261018')
if banknote_settings.get('RUN_BANKNOTE_EVAL','false').lower() == 'true':
    banknote_report = ROOT/'reports'/('evaluation_banknotes_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
    banknote_result = evaluate_banknote_pipeline(
        ROOT/banknote_settings['PRIMITIVE_CHECKPOINT_PATH'],ROOT/banknote_settings['BANKNOTE_CHECKPOINT_PATH'],
        ROOT/'data/annotations/events.csv', ROOT/'data/processed/split_proposal.csv',ROOT/'data/raw/videos',
        banknote_report,device=banknote_settings.get('BANKNOTE_DEVICE','cpu'),
        track_banknotes=banknote_settings.get('BANKNOTE_TRACKING','false').lower()=='true')
elif (banknote_report/'summary.json').exists():
    banknote_result=json.loads((banknote_report/'summary.json').read_text(encoding='utf-8'))
else:
    banknote_result=None; print('Отчёт отсутствует:',banknote_report)


In [ ]:
if banknote_result:
    display(pd.DataFrame(banknote_result['primitive_metrics']['per_class']))
    display(pd.Series(banknote_result['sampled_grid_proxy']['video'],name='Визуальный сценарий — только просмотренная сетка'))
    display(pd.Series(banknote_result['known_action_markers'],name='Сопоставление с известными action-отметками'))
    display(pd.Series(banknote_result['timing']))
    for metric in banknote_result['primitive_metrics']['per_class']:
        import numpy as np
        matrix=np.array(metric['confusion_matrix']); fig,ax=plt.subplots(figsize=(4,3)); ax.imshow(matrix,cmap='Blues')
        ax.set(xticks=[0,1],yticks=[0,1],xticklabels=['absent','present'],yticklabels=['absent','present'],xlabel='Prediction',ylabel='Reviewed target',title=metric['event_type'])
        for (i,j),value in np.ndenumerate(matrix):ax.text(j,i,str(value),ha='center',va='center',color='red')
        plt.show();plt.close(fig)
    banknote_errors=pd.DataFrame(json.loads((banknote_report/'primitive_errors.json').read_text(encoding='utf-8')))
    display(banknote_errors)
    display(FileLink(str(banknote_report.with_suffix('.zip'))))


In [ ]:
# Saved trajectories: no training, threshold tuning or repeated inference here.
banknote_clip_id='X-38'
banknote_scan_path=banknote_report/f'{banknote_clip_id}_scan.json'
if banknote_result and banknote_scan_path.exists():
    banknote_scan=json.loads(banknote_scan_path.read_text(encoding='utf-8'))
    trace=pd.DataFrame([{'time_sec':r['time_sec'],**r['probabilities'],
                       'standalone_cash_score':r['banknotes'].get('raw_detection_score',r['banknotes']['score'])}
                       for r in banknote_scan['observations']])
    trace.set_index('time_sec').plot(figsize=(12,3),ylim=(0,1));plt.title(banknote_clip_id);plt.show()
    display(pd.DataFrame(banknote_scan['events']))
    if banknote_settings.get('SHOW_BANKNOTE_ERRORS','false').lower()=='true' and not banknote_errors.empty:
        import numpy as np
        from matplotlib.patches import Rectangle
        from overfitted_underqualified.ml.dataset import load_records,decode_frames
        records={r.clip_id:r for r in load_records(ROOT/'data/annotations/events.csv',ROOT/'data/processed/split_proposal.csv',ROOT/'data/raw/videos')}
        predictions=json.loads((banknote_report/'primitive_predictions.json').read_text(encoding='utf-8'))
        for error in banknote_errors[banknote_errors.primitive=='banknotes_on_counter'].head(3).to_dict('records'):
            row=next(r for r in predictions if r['clip_id']==error['clip_id'] and r['center_sec']==error['time_sec'])
            rgb=decode_frames(records[error['clip_id']],np.array([error['time_sec']]))[0].permute(1,2,0).numpy()
            fig,ax=plt.subplots(figsize=(5,7));ax.imshow(rgb)
            for x1,y1,x2,y2 in row['banknotes']['boxes']:
                ax.add_patch(Rectangle((x1*rgb.shape[1],y1*rgb.shape[0]),(x2-x1)*rgb.shape[1],(y2-y1)*rgb.shape[0],fill=False,edgecolor='red'))
            ax.set_title(f"{error['clip_id']} {error['time_sec']}s {error['error']}");ax.axis('off');plt.show()


### Что можно утверждать
  Рамка показывает предсказание конкретного визуального объекта. Тип кандидата получается
  из истории приёма, ухода клиента и сохранения банкнот, а не из бинарной метки нарушения.
  Правило из каталога — ссылка для проверки применимости. Отсутствие детекции не доказывает
  пустой стол; владельца денег, проводку и завершение операции система не устанавливает.
  Зафиксированный эксперимент 18.10: детектор купюр имеет F1=0.80 на 39 отдельных
  проверенных validation-кадрах. Сопровождение дало X-38 @ 29 s; это 4 s до action-отметки
  33 s и 2 s до визуальной отметки 31 s. На четырёх просмотренных сетках TP=1, TN=3,
  FP=FN=0; остальные шесть validation-видео не считаются подтверждёнными отрицательными.
  Это минимальная проверка одного сценария, а не независимая оценка всех типов.
  Перед коммитом очистите outputs/counts/attachments: банковские кадры остаются локальными.


## Дополнительные X-видео: бинарное обучение и устойчивость
  Варианты наследуют метку/время и split оригинала из events.csv. Все четыре варианта
  train используются в новом бинарном эксперименте, validation-варианты — только
  после обучения. Копии одного родителя не считаются независимыми наблюдениями.
  S-сценарии сохраняются отдельно и не получают метки X-роликов автоматически.


In [ ]:
import json
import pandas as pd
from IPython.display import display,FileLink
augmentation_study_path=ROOT/'reports/binary_augmentation_study_20261018.json'
if augmentation_study_path.exists():
    augmentation_study=json.loads(augmentation_study_path.read_text(encoding='utf-8'))
    comparison=[]
    for model_key in ('previous_metrics','new_metrics'):
        for scope in ('original_video_metrics','transformed_video_metrics'):
            comparison.append({'model':model_key,'scope':scope,**augmentation_study[model_key][scope]})
    display(pd.DataFrame(comparison))
    display(pd.DataFrame(augmentation_study['new_metrics']['per_variant_video_metrics']).T)
    display(FileLink(str(ROOT/(augmentation_study['new_report']+'.zip'))))
    print('Бинарные окна вокруг отметок. Это не метрики локализации или распознавания конкретных типов.')
else:
    print('Сохранённое сравнение аугментаций ещё отсутствует.')


## Контроль числа обновлений и доли аугментаций
 Прежние 20 эпох дали 120 шагов на оригиналах против 560 на полном наборе копий.
 Теперь обе головы начинают одинаково и делают 120 шагов: оригиналы против
 смеси с вероятностью 80% оригиналов и 20% вариантов. Последний checkpoint
 фиксируется до validation; порог 0.5. Это исследование известной development
 validation, не независимый test и не обещание 100% на новых источниках.


In [ ]:
import json,os
from pathlib import Path
from dotenv import dotenv_values
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display,FileLink
matched_settings={**dotenv_values(ROOT/'.env'),**os.environ}
matched_report=ROOT/matched_settings.get('MATCHED_STUDY_REPORT','reports/matched_augmentation_study_20261018T025716_174675Z.json')


In [ ]:
matched_study=json.loads(matched_report.read_text(encoding='utf-8')) if matched_report.exists() else None
if matched_study:
    rows=[]
    for model_name,result in matched_study['results'].items():
        for scope in ('original_video_metrics','transformed_video_metrics'):
            rows.append({'model':model_name,'scope':scope,**result[scope]})
    display(pd.DataFrame(rows))
    print('Оба checkpoint зафиксированы до оценки; accuracy не подгонялась к100%.')
else:
    print('Отчёт контролируемого эксперимента отсутствует:',matched_report)


In [ ]:
if matched_study:
    from overfitted_underqualified.ml.plots import plot_confusion
    for model_name,result in matched_study['results'].items():
        print(model_name,'original validation');plot_confusion(result['original_video_metrics'])
        print(model_name,'40correlated transformed validation cases');plot_confusion(result['transformed_video_metrics'])
        display(pd.DataFrame(result['per_variant_video_metrics']).T)
        report_directory=Path(matched_study['reports'])/model_name
        display(pd.DataFrame(json.loads((report_directory/'errors.json').read_text(encoding='utf-8'))))
        display(FileLink(str(report_directory.with_suffix('.zip'))))
